Imports y dispositivo

In [1]:
import os, math, random, time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets, transforms, utils as vutils
import matplotlib.pyplot as plt

SEED = 1337
random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [5]:
import torch
print("Torch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("CUDA de PyTorch:", torch.version.cuda)
print("cuDNN:", torch.backends.cudnn.version())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


Torch: 2.5.1+cu121
CUDA disponible: True
CUDA de PyTorch: 12.1
cuDNN: 90100
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [ ]:
import torch

# Acelera kernels en GPUs modernas (conv/linear en FP32)
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")  # o "highest" si no te da miedo el redondeo

# cuDNN: auto-tuning (rápido), quítalo si buscas reproducibilidad estricta
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

print("CUDA OK:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0))
print("VRAM total (GB):", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))


Hiperparámetros

In [3]:
cfg = {
    "dataset": "CIFAR10",
    "data_root": "./data",
    "epochs": 30,        # 
    "batch_size": 128,
    "image_size": 32,    # CIFAR es 32x32
    "nc": 3,             # number of channels (RGB)
    "nz": 128,           # tamaño del vector latente z
    "ngf": 64,           # width del generador
    "ndf": 64,           # width del discriminador
    "lr": 2e-4,
    "beta1": 0.5,
    "beta2": 0.999,
    "num_workers": 2,
    "out_dir": "./runs_gan",
    "sample_n": 64
}
Path(cfg["out_dir"]).mkdir(parents=True, exist_ok=True)
cfg

{'dataset': 'CIFAR10',
 'data_root': './data',
 'epochs': 30,
 'batch_size': 128,
 'image_size': 32,
 'nc': 3,
 'nz': 128,
 'ngf': 64,
 'ndf': 64,
 'lr': 0.0002,
 'beta1': 0.5,
 'beta2': 0.999,
 'num_workers': 2,
 'out_dir': './runs_gan',
 'sample_n': 64}

DataLoader de CIFAR-10 (normalizado a [-1,1])

In [4]:
transform = transforms.Compose([
    transforms.Resize(cfg["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # de 0..1 a -1..1
])

trainset = datasets.CIFAR10(root=cfg["data_root"], train=True, download=True, transform=transform)
dloader = DataLoader(trainset, batch_size=cfg["batch_size"], shuffle=True, num_workers=cfg["num_workers"], drop_last=True)

# sanity check: ver batch shape
xb, yb = next(iter(dloader))
xb.shape, yb.shape, xb.min().item(), xb.max().item()

  9%|▉         | 15.8M/170M [00:05<00:52, 2.93MB/s]


KeyboardInterrupt: 

Inicialización de pesos (DCGAN)

In [ ]:
def weights_init_dcgan(m):
    classname = m.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if getattr(m, "bias", None) is not None and m.bias is not None:
            nn.init.zeros_(m.bias.data)
    elif "BatchNorm" in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.zeros_(m.bias.data)

Generador (DCGAN para 32×32, 3 canales)

In [ ]:
class Generator(nn.Module):
    def __init__(self, nz=128, ngf=64, nc=3):
        super().__init__()
        self.net = nn.Sequential(
            # input Z: (nz, 1, 1) -> (ngf*8, 4, 4)
            nn.ConvTranspose2d(nz, ngf*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf*8), nn.ReLU(True),

            # (ngf*8, 4, 4) -> (ngf*4, 8, 8)
            nn.ConvTranspose2d(ngf*8, ngf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*4), nn.ReLU(True),

            # (ngf*4, 8, 8) -> (ngf*2, 16, 16)
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2), nn.ReLU(True),

            # (ngf*2, 16, 16) -> (ngf, 32, 32)
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),

            # (ngf, 32, 32) -> (nc, 32, 32)
            nn.ConvTranspose2d(ngf, nc, 3, 1, 1, bias=False),
            nn.Tanh()  # salida en [-1,1]
        )

    def forward(self, z):
        return self.net(z)

Discriminador (DCGAN para 3×32×32)

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, ndf=64, nc=3):
        super().__init__()
        self.net = nn.Sequential(
            # (nc, 32, 32) -> (ndf, 16, 16)
            nn.Conv2d(nc, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # (ndf, 16, 16) -> (ndf*2, 8, 8)
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2),
            nn.LeakyReLU(0.2, inplace=True),

            # (ndf*2, 8, 8) -> (ndf*4, 4, 4)
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4),
            nn.LeakyReLU(0.2, inplace=True),

            # (ndf*4, 4, 4) -> (1, 1, 1)
            nn.Conv2d(ndf*4, 1, 4, 1, 0, bias=False)
        )

    def forward(self, x):
        # devolvemos logits (no aplicar Sigmoid aquí; usamos BCEWithLogitsLoss)
        return self.net(x).view(-1)

Instanciar modelos, criterio y optimizadores

In [ ]:
netG = Generator(cfg["nz"], cfg["ngf"], cfg["nc"]).to(device)
netD = Discriminator(cfg["ndf"], cfg["nc"]).to(device)
netG.apply(weights_init_dcgan)
netD.apply(weights_init_dcgan)

criterion = nn.BCEWithLogitsLoss()
optD = torch.optim.Adam(netD.parameters(), lr=cfg["lr"], betas=(cfg["beta1"], cfg["beta2"]))
optG = torch.optim.Adam(netG.parameters(), lr=cfg["lr"], betas=(cfg["beta1"], cfg["beta2"]))

fixed_noise = torch.randn(cfg["sample_n"], cfg["nz"], 1, 1, device=device)

sum(p.numel() for p in netG.parameters())/1e6, sum(p.numel() for p in netD.parameters())/1e6

Helpers de visualización y guardado

In [ ]:
def show_grid(tensor, nrow=8, title=None):
    grid = vutils.make_grid(tensor, nrow=nrow, normalize=True, value_range=(-1,1))
    plt.figure(figsize=(8,8)); plt.axis("off")
    if title: plt.title(title)
    plt.imshow(grid.permute(1,2,0).cpu().numpy())
    plt.show()

# muestra reales
real_batch, _ = next(iter(dloader))
show_grid(real_batch[:cfg["sample_n"]], title="Reales (CIFAR10)")

# muestra inicial de G
with torch.no_grad():
    fake0 = netG(fixed_noise).detach().cpu()
show_grid(fake0, title="Inicial (ruido)")

Loop de entrenamiento (DCGAN clásico)

In [ ]:
global_step = 0
start_time = time.time()
for epoch in range(cfg["epochs"]):
    for i, (real, _) in enumerate(dloader):
        real = real.to(device)

        # ====== (1) Actualiza D: maximiza log(D(x)) + log(1 - D(G(z)))
        netD.train(); netG.train()
        optD.zero_grad()

        bsz = real.size(0)
        lbl_real = torch.ones(bsz, device=device)
        lbl_fake = torch.zeros(bsz, device=device)

        # Real
        logits_real = netD(real)
        lossD_real = criterion(logits_real, lbl_real)

        # Fake
        z = torch.randn(bsz, cfg["nz"], 1, 1, device=device)
        fake = netG(z).detach()   # detach para no backprop a G aquí
        logits_fake = netD(fake)
        lossD_fake = criterion(logits_fake, lbl_fake)

        lossD = lossD_real + lossD_fake
        lossD.backward()
        optD.step()

        # ====== (2) Actualiza G: minimiza log(1 - D(G(z)))  <=>  maximiza log(D(G(z)))
        optG.zero_grad()
        z = torch.randn(bsz, cfg["nz"], 1, 1, device=device)
        gen = netG(z)
        logits_gen = netD(gen)
        lossG = criterion(logits_gen, lbl_real)
        lossG.backward()
        optG.step()

        global_step += 1
        if i % 100 == 0:
            print(f"[{epoch+1:03d}/{cfg['epochs']:03d}] iter {i:04d} | "
                  f"D: {lossD.item():.3f} (R {lossD_real.item():.3f} / F {lossD_fake.item():.3f}) | "
                  f"G: {lossG.item():.3f}")

    # samples por época
    with torch.no_grad():
        netG.eval()
        fakes = netG(fixed_noise).detach().cpu()
    vutils.save_image(fakes, os.path.join(cfg["out_dir"], f"epoch_{epoch+1:03d}.png"),
                      nrow=8, normalize=True, value_range=(-1,1))
    show_grid(fakes, title=f"Fakes @ epoch {epoch+1}")

    # checkpoints
    torch.save(netG.state_dict(), os.path.join(cfg["out_dir"], f"netG_{epoch+1:03d}.pt"))
    torch.save(netD.state_dict(), os.path.join(cfg["out_dir"], f"netD_{epoch+1:03d}.pt"))

print(f"Listo en {time.time()-start_time:.1f}s")